### В данном файле:
1) Используем тривиальный select. Для форматированного вывода пробуем tabulate
2) Обычный join-запрос, чтобы соединить в ответе данные двух таблиц. Для форматированного вывода используем pprint
3) Классический where-запрос. Для форматированного вывода используем json
4) Нехитрый запрос с группировкой, case-then и аггрегатной функцией. Для форматированного вывода используем display

In [ ]:
%pip install tabulate

In [2]:
from private.utils import cursor as cur
from tabulate import tabulate

with cur() as cursor:
    cursor.execute("SELECT id, personality_type, deep_reflection, created_at FROM user_info")
    
    # Fetch 10 rows from the executed query
    rows = cursor.fetchmany(10)
    # Получаем названия колонок из описания курсора
    headers = [desc[0] for desc in cursor.description]
    # Выводим таблицу
    print(tabulate(rows, headers=headers, tablefmt="grid"))
    

Successfully connected to the database!
+------+--------------------+-------------------+----------------------------+
|   id | personality_type   |   deep_reflection | created_at                 |
+======+====================+===================+============================+
|    1 | Extrovert          |           2.51515 | 2026-08-07 22:40:24.971068 |
+------+--------------------+-------------------+----------------------------+
|    2 | Ambivert           |           7.27449 | 2026-08-07 22:40:24.971068 |
+------+--------------------+-------------------+----------------------------+
|    3 | Ambivert           |           4.62226 | 2026-08-07 22:40:24.971068 |
+------+--------------------+-------------------+----------------------------+
|    4 | Extrovert          |           1.96544 | 2026-08-07 22:40:24.971068 |
+------+--------------------+-------------------+----------------------------+
|    5 | Introvert          |           9.92616 | 2026-08-07 22:40:24.971068 |
+------+----

In [3]:
from pprint import pprint

with cur() as cursor:
    cursor.execute('''
        SELECT 
            u.id, 
            u.personality_type, 
            u.social_energy,
            a.gadget_usage,
            a.online_social_usage
        FROM 
            user_info AS u
        INNER JOIN 
            additional_user_info AS a ON u.id = a.user_id
    ''')
    pprint(cursor.fetchmany(20))

Successfully connected to the database!
[(1, 'Extrovert', 6.794295229854372, 9.19171076842172, 9.154296028672762),
 (2, 'Ambivert', 6.378987653468172, 5.956141253087173, 4.683780661608601),
 (3, 'Ambivert', 7.459420713592305, 6.033047571604976, 5.000338154016388),
 (4, 'Extrovert', 6.159626485557139, 5.410144571831847, 7.601946338966451),
 (5, 'Introvert', 5.568461592550384, 5.704597885807435, 7.771568597293047),
 (6, 'Introvert', 2.807173448422305, 4.027298758951044, 3.4274457957040143),
 (7, 'Introvert', 1.5374679225922288, 4.056359012902543, 5.218916540802724),
 (8, 'Ambivert', 6.636632170312275, 7.147859415204639, 7.029383142831444),
 (9, 'Extrovert', 7.330318282617428, 8.788241116739812, 5.509409450828025),
 (10, 'Ambivert', 5.377394985952758, 3.5392729942143824, 5.644437964203616),
 (11, 'Introvert', 2.393582633681881, 4.208451676178816, 5.848730490105687),
 (12, 'Ambivert', 4.27489310220265, 6.417490175036427, 4.5134376538622085),
 (13, 'Ambivert', 6.585995548602856, 6.883344747

In [23]:
import json

with cur() as cursor:
    cursor.execute('''
        SELECT 
            id, 
            personality_type, 
            alone_time_preference, 
            listening_skill,
			curiosity
        FROM 
            public.user_info
        WHERE
            personality_type like '%tro%' 
			and (alone_time_preference > 9.9 or listening_skill > 9.9) 
			and curiosity < 1.5
    ''')

    cols = [desc[0] for desc in cursor.description]
    rows = cursor.fetchall()
    print(json.dumps([dict(zip(cols, row)) for row in rows], indent=4, ensure_ascii=False))

Successfully connected to the database!
[
    {
        "id": 3912,
        "personality_type": "Introvert",
        "alone_time_preference": 10.0,
        "listening_skill": 9.460963447893144,
        "curiosity": 1.461637290718639
    },
    {
        "id": 12769,
        "personality_type": "Introvert",
        "alone_time_preference": 9.442207903226585,
        "listening_skill": 10.0,
        "curiosity": 1.2621365689623207
    }
]


In [25]:
import pandas as pd

with cur() as cursor:
    cursor.execute('''
        SELECT 
            personality_type, 
            CASE 
                WHEN curiosity <= 5 THEN '0-5'
                ELSE '5-10'
            END as range,
            count(*) as curiosity
        FROM 
            user_info
        GROUP BY 
            personality_type, 
            CASE 
                WHEN curiosity <= 5 THEN '0-5'
                ELSE '5-10'
            END 
	    ORDER BY personality_type;
    ''')
    display(pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description]))  

Successfully connected to the database!


,personality_type,range,curiosity
0,Ambivert,0-5,1037
1,Ambivert,5-10,5536
2,Extrovert,0-5,616
3,Extrovert,5-10,6241
4,Introvert,0-5,1640
5,Introvert,5-10,4930
